# 🤖 AI-Powered Customer Support Agent with Memory & Tool Calling## Complete Google Colab Implementation (LangChain 1.2.0+)### 📋 Project OverviewThis notebook implements a production-ready AI customer support agent with:- **Memory System**: Persistent memory with Mem0 + Chroma vector store- **Tool Calling**: Dynamic tool execution for customer lookups, billing checks, etc.- **RAG Integration**: ChromaDB-based knowledge base retrieval- **Agent Orchestration**: LangGraph with checkpoint management- **API Endpoints**: FastAPI-compatible services### 🔑 Required API Keys & Environment VariablesYou'll need to set up the following in Google Colab Secrets:1. **GROQ_API_KEY** (Free tier available)   - Get from: https://console.groq.com   - Model: `llama-3.1-8b-instant` or `llama-3.1-70b-versatile`2. **TAVILY_API_KEY** (Optional - for web search)   - Get from: https://tavily.com3. **GOOGLE_API_KEY** (For embeddings - Optional)   - Get from: https://ai.google.dev   - Used for Gemini embeddings### 📦 Version Requirements- **LangChain**: >= 1.2.0- **LangGraph**: >= 0.1.0+- **LangChain Community**: >= 0.2.0+- **LangChain Core**: >= 0.2.0+- **Groq**: >= 0.7.0+- **ChromaDB**: >= 0.4.0+- **Mem0**: Latest- **FastAPI**: >= 0.104.0+### 🎯 Project Structure```/content/drive/MyDrive/customer_support_agent/├── data/│   ├── support.db                 # SQLite database│   ├── chroma_rag/               # RAG vector store│   └── chroma_mem0/              # Memory vector store├── knowledge_base/               # KB documents├── phase_1_setup.py             # Environment & dependencies├── phase_2_core.py              # Settings & configuration├── phase_3_integrations.py      # Tools, Memory, RAG├── phase_4_services.py          # Core services (Copilot, Knowledge)├── phase_5_api.py               # API routes & factory├── phase_6_main.py              # Application entry point└── DEPLOYMENT.md                # Step-by-step deployment```

## PHASE 1️⃣: Environment Setup & Dependencies Installation### 📥 Install Required PackagesAll packages are pinned to compatible versions. LangChain 1.2.0+ required.

In [ ]:
# PHASE 1: Install dependencies with specific versions for LangChain 1.2.0+import subprocessimport syspackages_to_install = [    # Core LangChain packages (>=1.2.0)    "langchain>=1.2.0",    "langchain-core>=0.2.0",    "langchain-community>=0.2.0",    "langgraph>=0.1.0",        # LLM & API clients    "langchain-groq>=0.1.0",    "groq>=0.7.0",        # Vector DB & Storage    "chromadb>=0.4.0",    "mem0ai>=0.1.0",        # Web Framework    "fastapi>=0.104.0",    "uvicorn[standard]>=0.24.0",        # Database    "sqlalchemy>=2.0",    "pydantic>=2.0",    "pydantic-settings>=2.0",        # Utilities    "python-dotenv>=1.0.0",    "requests>=2.31.0",    "email-validator>=2.1.0",    "google-genai>=0.3.0",        # Additional utilities    "tqdm>=4.66.0",]print("🔧 Installing dependencies for AI Customer Support Agent...")print("⏳ This may take 3-5 minutes...")for package in packages_to_install:    print(f"📦 Installing: {package}")    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])print("✅ All dependencies installed successfully!")print("📝 LangChain Version:", end=" ")import langchainprint(langchain.__version__)print("📝 LangGraph Version:", end=" ")import langgraphprint(langgraph.__version__)

### 🔐 Setup Google Colab SecretsRun the cell below to authenticate and set up your API keys in Colab Secrets:

In [ ]:
# Setup Colab Secrets and Environmentfrom google.colab import userdataimport osprint("🔐 Setting up API Keys from Google Colab Secrets...")print("=" * 60)try:    # Get GROQ API Key (Required)    groq_api_key = userdata.get('GROQ_API_KEY')    os.environ['GROQ_API_KEY'] = groq_api_key    print("✅ GROQ_API_KEY loaded successfully")except Exception as e:    print(f"⚠️  GROQ_API_KEY not found: {e}")    print("📌 Please add GROQ_API_KEY to Colab Secrets (click 🔑 icon)")try:    # Get Tavily API Key (Optional)    tavily_api_key = userdata.get('TAVILY_API_KEY')    os.environ['TAVILY_API_KEY'] = tavily_api_key    print("✅ TAVILY_API_KEY loaded successfully")except:    print("⚠️  TAVILY_API_KEY not configured (optional)")try:    # Get Google API Key (Optional)    google_api_key = userdata.get('GOOGLE_API_KEY')    os.environ['GOOGLE_API_KEY'] = google_api_key    print("✅ GOOGLE_API_KEY loaded successfully")except:    print("⚠️  GOOGLE_API_KEY not configured (optional)")print("=" * 60)print("✨ Environment variables configured!")

### 📁 Setup Project Directory StructureCreate the necessary directories in Google Drive:

In [ ]:
# Mount Google Drive and setup project structurefrom google.colab import drivefrom pathlib import Pathimport osimport jsonfrom datetime import datetime# Mount Google Drivedrive.mount('/content/drive', force_remount=True)# Create project base directoryproject_root = Path('/content/drive/MyDrive/customer_support_agent')project_root.mkdir(parents=True, exist_ok=True)# Create subdirectoriesdirectories = {    'data': project_root / 'data',    'chroma_rag': project_root / 'data' / 'chroma_rag',    'chroma_mem0': project_root / 'data' / 'chroma_mem0',    'knowledge_base': project_root / 'knowledge_base',    'modules': project_root / 'modules',    'outputs': project_root / 'outputs',}for name, path in directories.items():    path.mkdir(parents=True, exist_ok=True)    print(f"✅ Created: {path}")# Setup Python pathimport syssys.path.insert(0, str(project_root))# Create .env file templateenv_template = f"""# Customer Support Agent ConfigurationGROQ_API_KEY={os.environ.get('GROQ_API_KEY', 'your-key-here')}GROQ_MODEL=llama-3.1-8b-instantLLM_TEMPERATURE=0.2# Optional API KeysGOOGLE_API_KEY={os.environ.get('GOOGLE_API_KEY', '')}TAVILY_API_KEY={os.environ.get('TAVILY_API_KEY', '')}# Database ConfigurationDB_PATH=data/support.dbCHROMA_RAG_DIR=data/chroma_ragCHROMA_MEM0_DIR=data/chroma_mem0KNOWLEDGE_BASE_DIR=knowledge_base# RAG ConfigurationRAG_CHUNK_SIZE=800RAG_CHUNK_OVERLAP=120RAG_TOP_K=4MEM0_TOP_K=5# API ConfigurationAPI_HOST=0.0.0.0API_PORT=8000"""env_path = project_root / '.env'env_path.write_text(env_template)print(f"✅ Created .env template at: {env_path}")# Create metadata filemetadata = {    'created_at': datetime.now().isoformat(),    'project_root': str(project_root),    'lanchain_version': '1.2.0+',    'status': 'setup_complete'}metadata_path = project_root / 'metadata.json'with open(metadata_path, 'w') as f:    json.dump(metadata, f, indent=2)print(f"📊 Project Structure Created:")print(f"   Root: {project_root}")for name, path in directories.items():    print(f"   └── {name}: {path}")os.chdir(project_root)print(f"✅ Changed working directory to: {os.getcwd()}")

## PHASE 2️⃣: Core Configuration & Settings### 🔧 Define Settings & Configuration ClassesLangChain 1.2.0+ compatible Pydantic models for configuration management.

In [ ]:
# PHASE 2: Core Settings Configuration (LangChain 1.2.0+ compatible)from pydantic_settings import BaseSettings, SettingsConfigDictfrom pydantic import Fieldfrom pathlib import Pathfrom functools import lru_cachefrom typing import Optionalimport osfrom dotenv import load_dotenv# Load environment variablesload_dotenv('.env', override=True)class Settings(BaseSettings):    """    Application settings with environment variable support.    Compatible with LangChain 1.2.0+    """        model_config = SettingsConfigDict(        env_file=".env",        env_file_encoding="utf-8",        extra="ignore",        case_sensitive=False    )        # ==================== APP CONFIG ====================    app_name: str = Field(        default="AI Copilot for Support Agents",        description="Application name"    )    app_version: str = Field(        default="1.0.0",        description="Application version"    )        # ==================== LLM CONFIG ====================    groq_api_key: str = Field(        default="",        description="Groq API Key"    )    groq_model: str = Field(        default="llama-3.1-8b-instant",        description="Groq model to use (8b-instant or 70b-versatile)"    )    llm_temperature: float = Field(        default=0.2,        description="LLM temperature for generation"    )    llm_max_tokens: int = Field(        default=2048,        description="Maximum tokens for LLM output"    )        # ==================== EMBEDDING CONFIG ====================    google_api_key: Optional[str] = Field(        default=None,        description="Google API Key for embeddings"    )    google_embedding_model: str = Field(        default="gemini-embedding-001",        description="Google embedding model"    )    enable_local_embeddings: bool = Field(        default=False,        description="Use local embeddings (sentence-transformers)"    )        # ==================== STORAGE PATHS ====================    workspace_dir: Path = Field(        default_factory=lambda: Path.cwd(),        description="Workspace directory"    )    data_dir: Path = Field(        default=Path("data"),        description="Data directory"    )    db_path: Path = Field(        default=Path("data/support.db"),        description="SQLite database path"    )    chroma_rag_dir: Path = Field(        default=Path("data/chroma_rag"),        description="ChromaDB RAG directory"    )    chroma_mem0_dir: Path = Field(        default=Path("data/chroma_mem0"),        description="ChromaDB Mem0 directory"    )    knowledge_base_dir: Path = Field(        default=Path("knowledge_base"),        description="Knowledge base documents directory"    )        # ==================== RAG CONFIG ====================    rag_chunk_size: int = Field(        default=800,        description="Chunk size for RAG document splitting"    )    rag_chunk_overlap: int = Field(        default=120,        description="Chunk overlap for RAG"    )    rag_top_k: int = Field(        default=4,        description="Top K documents to retrieve from RAG"    )    mem0_top_k: int = Field(        default=5,        description="Top K memories to retrieve"    )        # ==================== API CONFIG ====================    api_host: str = Field(        default="0.0.0.0",        description="API host"    )    api_port: int = Field(        default=8000,        description="API port"    )    dashboard_api_url: str = Field(        default="http://localhost:8000",        description="Dashboard API URL"    )        def resolve(self, path: Path) -> Path:        """Resolve relative paths against the project root."""        return path if path.is_absolute() else self.workspace_dir / path        @property    def db_file(self) -> Path:        """Get resolved database file path."""        return self.resolve(self.db_path)        @property    def chroma_rag_path(self) -> Path:        """Get resolved ChromaDB RAG path."""        return self.resolve(self.chroma_rag_dir)        @property    def chroma_mem0_path(self) -> Path:        """Get resolved ChromaDB Mem0 path."""        return self.resolve(self.chroma_mem0_dir)        @property    def knowledge_base_path(self) -> Path:        """Get resolved knowledge base path."""        return self.resolve(self.knowledge_base_dir)        @property    def effective_google_embedding_model(self) -> str:        """        Normalize and auto-upgrade legacy embedding model IDs to a supported Gemini model.        """        model = (self.google_embedding_model or "").strip()        if not model:            return "gemini-embedding-001"                if model.startswith("models/"):            model = model[len("models/"):]                deprecated_aliases = {            "text-embedding-004",            "embedding-001",            "embedding-gecko-001",            "gemini-embedding-exp",            "gemini-embedding-exp-03-07",        }        if model in deprecated_aliases:            return "gemini-embedding-001"                return model@lru_cache(maxsize=1)def get_settings() -> Settings:    """    Get cached settings instance.    Uses LRU cache to ensure single instance across application.    """    return Settings()def ensure_directories(settings: Optional[Settings] = None) -> None:    """    Create all required directories.        Args:        settings: Optional Settings instance (uses default if None)    """    config = settings or get_settings()        directories = [        config.resolve(config.data_dir),        config.chroma_rag_path,        config.chroma_mem0_path,        config.knowledge_base_path,    ]        for path in directories:        path.mkdir(parents=True, exist_ok=True)        print(f"✅ Ensured directory: {path}")# Test configurationprint("\n🔧 Testing Configuration Setup...")settings = get_settings()print(f"✅ App Name: {settings.app_name}")print(f"✅ Groq Model: {settings.groq_model}")print(f"✅ Temperature: {settings.llm_temperature}")print(f"✅ Workspace: {settings.workspace_dir}")# Ensure directories existensure_directories(settings)print("\n✅ Configuration Phase Complete!")

## PHASE 3️⃣: Integrations (Tools, Memory, RAG)### 🛠️ Support Tools for Customer LookupThese tools are called by the agent for real-time customer data lookups.

In [ ]:
# PHASE 3A: Support Tools (LangChain 1.2.0+ compatible)import hashlibimport jsonfrom typing import Anyfrom langchain_core.tools import toolimport datetimedef _stable_bucket(email: str, size: int) -> int:    """    Create stable bucket assignment based on email hash.    Ensures consistent tool output for same customer.    """    digest = hashlib.sha256(email.strip().lower().encode("utf-8")).hexdigest()    return int(digest, 16) % sizedef _json(payload: dict[str, Any]) -> str:    """Serialize payload to JSON string."""    return json.dumps(payload, indent=2)def _load_band(open_count: int) -> str:    """Categorize customer load based on open tickets."""    if open_count <= 1:        return "light"    if open_count <= 3:        return "moderate"    return "heavy"# ==================== TOOL 1: Lookup Customer Plan ====================@tool(name="lookup_customer_plan")def lookup_customer_plan(customer_email: str) -> str:    """    Return structured subscription and SLA details for a customer email.        Args:        customer_email: Customer email address        Returns:        JSON string with plan details, SLA hours, and priority queue status    """    plans = [        {"plan_tier": "free", "sla_hours": 48, "priority_queue": False},        {"plan_tier": "starter", "sla_hours": 24, "priority_queue": False},        {"plan_tier": "pro", "sla_hours": 8, "priority_queue": True},        {"plan_tier": "enterprise", "sla_hours": 1, "priority_queue": True},    ]        plan = plans[_stable_bucket(customer_email, len(plans))]        summary = (        f"{customer_email} is on the {plan['plan_tier']} plan with "        f"{plan['sla_hours']}h SLA."    )        return _json({        "tool": "lookup_customer_plan",        "customer_email": customer_email,        "summary": summary,        "details": plan,        "recommended_action": (            "Prioritize this ticket in the support queue"            if plan['priority_queue']            else "Handle in normal order"        ),    })# ==================== TOOL 2: Check Billing Status ====================@tool(name="check_billing_status")def check_billing_status(customer_email: str) -> str:    """    Check payment and billing status for a customer.        Args:        customer_email: Customer email address        Returns:        JSON string with billing status and risk level    """    risk_levels = ["ok", "warning", "high_risk", "suspended"]    risk_level = risk_levels[_stable_bucket(customer_email, len(risk_levels))]        is_overdue = risk_level in ["high_risk", "suspended"]    summary = (        f"Billing status for {customer_email}: {risk_level.upper()}"    )    if is_overdue:        summary += " - Account has overdue payments"        return _json({        "tool": "check_billing_status",        "customer_email": customer_email,        "summary": summary,        "details": {            "status": risk_level,            "overdue": is_overdue,            "last_payment_date": "2024-05-15",            "next_billing_date": "2024-06-15",        },        "recommended_action": (            "Flag for billing team before resolution"            if is_overdue            else "Proceed with standard support"        ),    })# ==================== TOOL 3: Get Ticket History ====================@tool(name="get_ticket_history")def get_ticket_history(customer_email: str) -> str:    """    Retrieve recent ticket history for a customer.        Args:        customer_email: Customer email address        Returns:        JSON string with recent tickets and patterns    """    open_tickets = _stable_bucket(customer_email, 5)    resolved_tickets = _stable_bucket(customer_email, 10) + 10        summary = (        f"{customer_email} has {open_tickets} open tickets and "        f"{resolved_tickets} resolved tickets"    )        return _json({        "tool": "get_ticket_history",        "customer_email": customer_email,        "summary": summary,        "details": {            "open_tickets": open_tickets,            "resolved_tickets": resolved_tickets,            "load_band": _load_band(open_tickets),            "avg_resolution_time_hours": 12 + (open_tickets * 4),        },        "recommended_action": (            "Check for recurring issues"            if open_tickets > 2            else "Standard support workflow"        ),    })# ==================== TOOL 4: Lookup Account Details ====================@tool(name="lookup_account_details")def lookup_account_details(customer_email: str) -> str:    """    Get comprehensive account information for a customer.        Args:        customer_email: Customer email address        Returns:        JSON string with account metadata    """    account_types = ["trial", "standard", "verified", "vip"]    account_type = account_types[_stable_bucket(customer_email, len(account_types))]        summary = (        f"{customer_email} has {account_type} account status"    )        return _json({        "tool": "lookup_account_details",        "customer_email": customer_email,        "summary": summary,        "details": {            "account_type": account_type,            "created_date": "2023-08-15",            "verified": account_type in ["verified", "vip"],            "two_factor_enabled": account_type in ["verified", "vip"],            "api_quota_usage": _stable_bucket(customer_email, 100),        },        "recommended_action": (            "Offer account upgrade"            if account_type == "trial"            else "Provide account insights"        ),    })def get_support_tools() -> list:    """    Get all available support tools.    Compatible with LangChain 1.2.0+ tool format.        Returns:        List of callable tool objects    """    return [        lookup_customer_plan,        check_billing_status,        get_ticket_history,        lookup_account_details,    ]# Test toolsprint("\n🛠️ Testing Support Tools...")tools = get_support_tools()print(f"✅ Loaded {len(tools)} support tools:")for tool in tools:    print(f"   - {tool.name}: {tool.description}")# Test a tool executionprint("\n🧪 Testing Tool Execution:")result = lookup_customer_plan("customer@example.com")print(f"✅ Tool result: {result}")print("\n✅ Support Tools Phase Complete!")

### 💾 Memory System Integration (Mem0 + ChromaDB)Persistent memory store for customer interactions and resolutions.

In [ ]:
# PHASE 3B: Memory System (Mem0 + ChromaDB - LangChain 1.2.0+ compatible)from typing import Optional, Anyfrom langchain_groq import ChatGroqimport jsonfrom pathlib import Pathclass CustomerMemoryStore:    """    Customer memory store using Mem0 and ChromaDB.    Stores resolutions, interactions, and patterns for each customer.        Compatible with LangChain 1.2.0+    """        def __init__(self, settings: Settings, llm: ChatGroq):        """        Initialize memory store.                Args:            settings: Application settings            llm: LangChain LLM instance (ChatGroq)        """        self.settings = settings        self.llm = llm                # Initialize ChromaDB for memory storage        try:            import chromadb                        self.client = chromadb.PersistentClient(                path=str(settings.chroma_mem0_path)            )            self.collection = self.client.get_or_create_collection(                name="customer_memories",                metadata={"hnsw:space": "cosine"}            )            print(f"✅ ChromaDB Memory Collection initialized at: {settings.chroma_mem0_path}")        except Exception as e:            print(f"⚠️  Error initializing ChromaDB: {e}")            raise        def add_resolution(        self,        user_id: str,        ticket_subject: str,        ticket_description: str,        accepted_draft: str,        entity_links: Optional[list[str]] = None,    ) -> None:        """        Store an accepted resolution in memory.                Args:            user_id: Customer email or company scope ID            ticket_subject: Support ticket subject            ticket_description: Ticket description            accepted_draft: Final resolution text sent to customer            entity_links: Extracted entities (plans, regions, integrations)        """        memory_text = f"Subject: {ticket_subject}\nDescription: {ticket_description}\nResolution: {accepted_draft}"                metadata = {            "user_id": user_id,            "subject": ticket_subject,            "type": "resolution",            "entity_links": json.dumps(entity_links or []),        }                try:            # Add to ChromaDB collection            import uuid            self.collection.add(                ids=[str(uuid.uuid4())],                documents=[memory_text],                metadatas=[metadata],            )            print(f"✅ Stored resolution for: {user_id}")        except Exception as e:            print(f"⚠️  Error storing resolution: {e}")        def search(        self,        query: str,        user_id: str,        limit: int = 5,    ) -> list[dict[str, Any]]:        """        Search customer memories using semantic similarity.                Args:            query: Search query            user_id: Customer email or company scope ID            limit: Maximum results to return                Returns:            List of matching memories with metadata        """        try:            results = self.collection.query(                query_texts=[query],                n_results=limit,                where={"user_id": user_id} if user_id else None,            )                        memories = []            for i, doc in enumerate(results.get('documents', [[]])[0]):                memory = {                    "id": results.get('ids', [[]])[0][i] if results.get('ids') else None,                    "memory": doc,                    "metadata": results.get('metadatas', [[]])[0][i] if results.get('metadatas') else {},                    "distance": results.get('distances', [[]])[0][i] if results.get('distances') else 0,                }                memories.append(memory)                        return memories        except Exception as e:            print(f"⚠️  Error searching memories: {e}")            return []        def list_memories(        self,        user_id: str,        limit: int = 20,    ) -> list[dict[str, Any]]:        """        List all memories for a user.                Args:            user_id: Customer email or company scope ID            limit: Maximum results to return                Returns:            List of user memories        """        try:            results = self.collection.get(                where={"user_id": user_id},                limit=limit,            )                        memories = []            for i, doc in enumerate(results.get('documents', [])):                memory = {                    "id": results.get('ids', [])[i],                    "memory": doc,                    "metadata": results.get('metadatas', [])[i],                }                memories.append(memory)                        return memories        except Exception as e:            print(f"⚠️  Error listing memories: {e}")            return []        def delete_memory(self, memory_id: str) -> bool:        """        Delete a specific memory by ID.                Args:            memory_id: Memory ID to delete                Returns:            True if successful        """        try:            self.collection.delete(ids=[memory_id])            return True        except Exception as e:            print(f"⚠️  Error deleting memory: {e}")            return False# Test Memory Systemprint("\n💾 Testing Memory System...")settings = get_settings()llm = ChatGroq(    model=settings.groq_model,    groq_api_key=settings.groq_api_key,    temperature=settings.llm_temperature,)try:    memory_store = CustomerMemoryStore(settings=settings, llm=llm)    print("✅ Memory store initialized successfully!")        # Test storing a resolution    memory_store.add_resolution(        user_id="test@example.com",        ticket_subject="Billing issue",        ticket_description="Customer was charged twice",        accepted_draft="We've issued a refund and verified the issue won't recur.",        entity_links=["billing_risk:high", "plan:pro"]    )        print("✅ Test resolution stored successfully!")except Exception as e:    print(f"⚠️  Error in memory system: {e}")print("\n✅ Memory System Phase Complete!")

### 📚 RAG System (Knowledge Base Retrieval)ChromaDB-based knowledge base retrieval system.

In [ ]:
# PHASE 3C: RAG System (ChromaDB - LangChain 1.2.0+ compatible)from typing import Optional, Anyfrom pathlib import Pathimport jsonclass KnowledgeBaseService:    """    Knowledge Base Service using ChromaDB for RAG (Retrieval-Augmented Generation).        Stores and retrieves knowledge base documents to augment LLM responses.    Compatible with LangChain 1.2.0+    """        def __init__(self, settings: Settings):        """        Initialize knowledge base service.                Args:            settings: Application settings        """        self.settings = settings                try:            import chromadb                        self.client = chromadb.PersistentClient(                path=str(settings.chroma_rag_path)            )            self.collection = self.client.get_or_create_collection(                name="knowledge_base",                metadata={"hnsw:space": "cosine"}            )            print(f"✅ ChromaDB RAG Collection initialized at: {settings.chroma_rag_path}")        except Exception as e:            print(f"⚠️  Error initializing ChromaDB RAG: {e}")            raise        def add_knowledge(        self,        doc_id: str,        content: str,        source: str,        metadata: Optional[dict[str, Any]] = None,    ) -> None:        """        Add document to knowledge base.                Args:            doc_id: Unique document ID            content: Document content            source: Source/filename            metadata: Additional metadata        """        try:            meta = metadata or {}            meta.update({"source": source})                        self.collection.add(                ids=[doc_id],                documents=[content],                metadatas=[meta],            )            print(f"✅ Added knowledge document: {source}")        except Exception as e:            print(f"⚠️  Error adding knowledge: {e}")        def search(        self,        query: str,        top_k: int = 4,    ) -> list[dict[str, Any]]:        """        Search knowledge base using semantic similarity.                Args:            query: Search query            top_k: Number of results to return                Returns:            List of matching documents        """        try:            results = self.collection.query(                query_texts=[query],                n_results=top_k,            )                        documents = []            for i, doc in enumerate(results.get('documents', [[]])[0]):                doc_item = {                    "id": results.get('ids', [[]])[0][i],                    "content": doc,                    "source": results.get('metadatas', [[]])[0][i].get('source', 'unknown') if results.get('metadatas') else 'unknown',                    "distance": results.get('distances', [[]])[0][i] if results.get('distances') else 0,                    "metadata": results.get('metadatas', [[]])[0][i] if results.get('metadatas') else {},                }                documents.append(doc_item)                        return documents        except Exception as e:            print(f"⚠️  Error searching knowledge base: {e}")            return []        def load_documents_from_folder(self) -> int:        """        Load markdown documents from knowledge base folder.                Returns:            Number of documents loaded        """        kb_path = self.settings.knowledge_base_path                if not kb_path.exists():            print(f"⚠️  Knowledge base path does not exist: {kb_path}")            return 0                doc_count = 0        for file_path in kb_path.glob("*.md"):            try:                content = file_path.read_text(encoding='utf-8')                doc_id = f"kb_{file_path.stem}"                self.add_knowledge(                    doc_id=doc_id,                    content=content,                    source=file_path.name,                    metadata={"type": "knowledge_base"}                )                doc_count += 1            except Exception as e:                print(f"⚠️  Error loading {file_path.name}: {e}")                print(f"✅ Loaded {doc_count} documents from knowledge base")        return doc_count# Test RAG Systemprint("\n📚 Testing RAG System...")settings = get_settings()try:    rag_service = KnowledgeBaseService(settings=settings)    print("✅ RAG service initialized successfully!")        # Test adding knowledge    rag_service.add_knowledge(        doc_id="test_1",        content="Password reset instructions: Click 'Forgot Password' on login page...",        source="password_reset.md",        metadata={"category": "account"}    )    print("✅ Test knowledge added successfully!")        # Test searching    results = rag_service.search("How to reset password?", top_k=1)    print(f"✅ Search returned {len(results)} results")except Exception as e:    print(f"⚠️  Error in RAG system: {e}")print("\n✅ RAG System Phase Complete!")

## PHASE 4️⃣: Core Services (Copilot & Draft Generation)### 🤖 Support Copilot ServiceMain orchestration service that combines tools, memory, and RAG.

In [ ]:
# PHASE 4: Support Copilot Service (LangChain 1.2.0+ with LangGraph)import reimport jsonfrom typing import Any, Optionalfrom langchain.agents import create_agentfrom langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessagefrom langchain_groq import ChatGroqfrom langgraph.checkpoint.memory import InMemorySaverclass SupportCopilot:    """    AI-powered support copilot using LangChain 1.2.0+ and LangGraph.        Integrates:    - Tool calling for customer lookups    - Memory system for past resolutions    - RAG for knowledge base    - LangGraph agent with checkpoint management        Compatible with LangChain 1.2.0+ and latest versions.    """        def __init__(self, settings: Settings):        """        Initialize the support copilot.                Args:            settings: Application settings                Raises:            RuntimeError: If GROQ_API_KEY is not configured        """        if not settings.groq_api_key:            raise RuntimeError(                "GROQ_API_KEY is missing. Add it in environment before generating drafts."            )                self._settings = settings                # Initialize LLM (LangChain 1.2.0+ compatible)        self._llm = ChatGroq(            model=settings.groq_model,            groq_api_key=settings.groq_api_key,            temperature=settings.llm_temperature,            max_tokens=settings.llm_max_tokens,        )                # Get support tools (LangChain 1.2.0+ compatible)        self._tools = get_support_tools()                # Create agent with LangGraph checkpointer        self._agent = create_agent(            model=self._llm,            tools=self._tools,            checkpointer=InMemorySaver(),            name="support_copilot_agent",        )                # Initialize memory system        self._memory_error: str | None = None        try:            self.memory = CustomerMemoryStore(settings=settings, llm=self._llm)        except Exception as exc:            self._memory_error = str(exc)            self.memory = None                # Initialize RAG system        self.rag = KnowledgeBaseService(settings=settings)        def generate_draft(        self,        ticket: dict[str, Any],        customer: dict[str, Any]    ) -> dict[str, Any]:        """        Generate a customer support draft response.                Args:            ticket: Support ticket with 'subject' and 'description'            customer: Customer info with 'email', 'name', 'company'                Returns:            Dictionary with 'draft' (response text) and 'context_used' (metadata)        """        # Build query from ticket        query = f"{ticket.get('subject', '')}\n{ticket.get('description', '')}"        customer_email = customer.get("email", "unknown")                # Search memory and knowledge base        memory_hits = self._search_memory_scopes(            query=query,            customer_email=customer_email,            customer_company=customer.get("company"),            limit=self._settings.mem0_top_k,        )        kb_hits = self.rag.search(query=query, top_k=self._settings.rag_top_k)                # Build prompts with context        system_prompt = self._build_system_prompt(memory_hits=memory_hits, kb_hits=kb_hits)        user_prompt = self._build_user_prompt(ticket=ticket, customer=customer)                # Invoke agent with LangGraph        agent_result = self._agent.invoke(            {                "messages": [                    SystemMessage(content=system_prompt),                    HumanMessage(content=user_prompt),                ]            },            config={                "configurable": {                    "thread_id": self._thread_id_for_ticket(ticket=ticket, customer=customer),                },                "recursion_limit": 40,            },        )                # Extract draft and tool calls        draft_text, tool_calls = self._extract_agent_draft_and_tool_calls(agent_result)                # Fallback mechanisms if draft generation fails        used_fallback = False        if not draft_text:            draft_text = self._fallback_generate_text(                ticket=ticket,                customer=customer,                memory_hits=memory_hits,                kb_hits=kb_hits,                tool_calls=tool_calls,            )            used_fallback = True                if not draft_text:            draft_text = self._deterministic_fallback(                ticket=ticket,                customer=customer,                tool_calls=tool_calls            )            used_fallback = True                # Build context metadata        context_used = self._build_context(            ticket=ticket,            customer=customer,            memory_hits=memory_hits,            kb_hits=kb_hits,            tool_calls=tool_calls,        )                if self._memory_error:            context_used.setdefault("errors", []).append(                f"Memory disabled: {self._memory_error}"            )                if used_fallback:            context_used.setdefault("errors", []).append(                "Fallback synthesis was used for draft generation"            )                context_used["agent_runtime"] = "langchain_1_2_0_with_langgraph"                return {            "draft": draft_text,            "context_used": context_used,        }        # ==================== Helper Methods ====================        def _build_system_prompt(        self,        memory_hits: list[dict[str, Any]],        kb_hits: list[dict[str, Any]],    ) -> str:        """Build system prompt with memory and knowledge context."""        memory_context = ""        if memory_hits:            memory_context = "\n\nRELATED PAST RESOLUTIONS:\n"            for hit in memory_hits[:3]:                memory_context += f"- {hit.get('memory', '')}\n"                kb_context = ""        if kb_hits:            kb_context = "\n\nRELEVANT KNOWLEDGE BASE ARTICLES:\n"            for hit in kb_hits[:3]:                kb_context += f"- [{hit.get('source', 'unknown')}] {hit.get('content', '')}\n"                return (            "You are an expert customer support copilot for a software company. "            "Your job is to draft empathetic, professional customer support responses. "            "Use the available tools to gather customer information, and craft responses "            "that address the issue thoroughly with clear next steps. "            f"{memory_context}{kb_context}"        )        def _build_user_prompt(        self,        ticket: dict[str, Any],        customer: dict[str, Any],    ) -> str:        """Build user prompt with ticket and customer details."""        return (            f"Customer Name: {customer.get('name', 'Unknown')}\n"            f"Customer Email: {customer.get('email', 'unknown')}\n"            f"Customer Company: {customer.get('company', 'Unknown')}\n"            f"\n"            f"TICKET SUBJECT: {ticket.get('subject', 'No subject')}\n"            f"TICKET DESCRIPTION: {ticket.get('description', 'No description')}\n"            f"TICKET PRIORITY: {ticket.get('priority', 'Normal')}\n"            f"\n"            f"Please use the available tools to gather customer information, "            f"then draft a professional response that addresses their issue."        )        def _thread_id_for_ticket(        self,        ticket: dict[str, Any],        customer: dict[str, Any],    ) -> str:        """Generate consistent thread ID for LangGraph checkpointing."""        ticket_id = str(ticket.get('id', 'unknown'))        customer_email = customer.get('email', 'unknown')        return f"{customer_email}_{ticket_id}"        def _extract_agent_draft_and_tool_calls(        self,        agent_result: Any,    ) -> tuple[str, list[dict[str, Any]]]:        """Extract draft text and tool calls from agent result."""        draft_text = ""        tool_calls = []                messages = agent_result.get("messages", [])        for msg in messages:            if isinstance(msg, AIMessage):                draft_text = self._extract_content(msg)                                # Extract tool calls if present                if hasattr(msg, 'tool_calls') and msg.tool_calls:                    for tool_call in msg.tool_calls:                        tool_calls.append({                            "tool": tool_call.get('name', 'unknown'),                            "status": "ok",                        })                return draft_text, tool_calls        @staticmethod    def _extract_content(message: BaseMessage) -> str:        """Extract text content from BaseMessage."""        if isinstance(message, AIMessage):            if isinstance(message.content, str):                return message.content            elif isinstance(message.content, list):                for item in message.content:                    if isinstance(item, dict) and item.get("type") == "text":                        return item.get("text", "")        return ""        def _search_memory_scopes(        self,        query: str,        customer_email: str,        customer_company: Optional[str],        limit: int,    ) -> list[dict[str, Any]]:        """Search memory for both customer and company scopes."""        if not self.memory:            return []                raw_hits: list[dict[str, Any]] = []        scope_user_ids = self._memory_scope_ids(            customer_email=customer_email,            customer_company=customer_company,        )                for scope_user_id in scope_user_ids:            hits = self.memory.search(query=query, user_id=scope_user_id, limit=limit)            raw_hits.extend(self._annotate_memory_scope(hits=hits, scope_user_id=scope_user_id))                return self._dedupe_memory_hits(raw_hits, limit=limit)        @staticmethod    def _memory_scope_ids(        customer_email: str,        customer_company: Optional[str]    ) -> list[str]:        """Get memory scope IDs for customer and company."""        scope_user_ids = [customer_email.strip().lower()]                company_scope = SupportCopilot._company_scope_user_id(customer_company)        if company_scope:            scope_user_ids.append(company_scope)                return SupportCopilot._unique_ordered(scope_user_ids)        @staticmethod    def _company_scope_user_id(customer_company: Optional[str]) -> Optional[str]:        """Normalize company name to scope ID."""        if not customer_company:            return None                lowered = customer_company.strip().lower()        if not lowered:            return None                normalized = re.sub(r"[^a-z0-9]+", "-", lowered).strip("-")        if not normalized:            return None                return f"company::{normalized}"        @staticmethod    def _annotate_memory_scope(        hits: list[dict[str, Any]],        scope_user_id: str,    ) -> list[dict[str, Any]]:        """Annotate memory hits with scope information."""        annotated: list[dict[str, Any]] = []        scope = "company" if scope_user_id.startswith("company::") else "customer"                for hit in hits:            item = dict(hit)            metadata = dict(item.get("metadata") or {})            metadata.setdefault("scope", scope)            item["metadata"] = metadata            annotated.append(item)                return annotated        @staticmethod    def _unique_ordered(values: list[str]) -> list[str]:        """Remove duplicates while preserving order."""        seen: set[str] = set()        ordered: list[str] = []                for value in values:            if value in seen:                continue            seen.add(value)            ordered.append(value)                return ordered        @staticmethod    def _dedupe_memory_hits(        raw_hits: list[dict[str, Any]],        limit: int    ) -> list[dict[str, Any]]:        """Remove duplicate memory hits."""        seen: set[str] = set()        deduped: list[dict[str, Any]] = []                for hit in raw_hits:            hit_id = hit.get("id")            if hit_id in seen:                continue            seen.add(hit_id)            deduped.append(hit)                        if len(deduped) >= limit:                break                return deduped        def _build_context(        self,        ticket: dict[str, Any],        customer: dict[str, Any],        memory_hits: list[dict[str, Any]],        kb_hits: list[dict[str, Any]],        tool_calls: list[dict[str, Any]],    ) -> dict[str, Any]:        """Build context metadata for response."""        knowledge_sources = self._unique_ordered(            [str(item.get("source")) for item in kb_hits if item.get("source")]        )                return {            "version": 2,            "ticket": {                "id": ticket.get("id"),                "subject": ticket.get("subject"),                "priority": ticket.get("priority"),                "status": ticket.get("status"),            },            "customer": {                "id": customer.get("id"),                "email": customer.get("email"),                "name": customer.get("name"),                "company": customer.get("company"),            },            "signals": {                "memory_hit_count": len(memory_hits),                "knowledge_hit_count": len(kb_hits),                "tool_call_count": len(tool_calls),                "knowledge_sources": knowledge_sources,            },            "memory_hits": memory_hits,            "knowledge_hits": kb_hits,            "tool_calls": tool_calls,        }        def _fallback_generate_text(        self,        ticket: dict[str, Any],        customer: dict[str, Any],        memory_hits: list[dict[str, Any]],        kb_hits: list[dict[str, Any]],        tool_calls: list[dict[str, Any]],    ) -> str:        """Fallback draft generation using LLM without tools."""        try:            memory_context = ""            if memory_hits:                memory_context = "\nPast resolutions:\n"                for hit in memory_hits[:2]:                    memory_context += f"- {hit.get('memory', '')}\n"                        kb_context = ""            if kb_hits:                kb_context = "\nRelevant docs:\n"                for hit in kb_hits[:2]:                    kb_context += f"- {hit.get('content', '')}\n"                        user_message = (                f"Generate a support response for {customer.get('name', 'the customer')} "                f"regarding: {ticket.get('subject', 'their issue')}\n"                f"{memory_context}{kb_context}"            )                        response = self._llm.invoke([                SystemMessage(content="You are a helpful support agent. Write concise, empathetic responses."),                HumanMessage(content=user_message),            ])                        return self._extract_content(response).strip()        except Exception as e:            print(f"⚠️  Fallback generation failed: {e}")            return ""        def _deterministic_fallback(        self,        ticket: dict[str, Any],        customer: dict[str, Any],        tool_calls: list[dict[str, Any]],    ) -> str:        """Last-resort deterministic fallback response."""        customer_name = customer.get("name") or customer.get("email") or "there"                return (            f"Hi {customer_name},\n\n"            f"Thank you for reaching out about \"{ticket.get('subject', 'your issue')}\". "            "We understand how important this is to you.\n\n"            "Our support team is reviewing your account and the details you provided. "            "We will investigate thoroughly and provide a detailed update within 24 hours.\n\n"            "Best regards,\nSupport Team"        )# Test Copilotprint("\n🤖 Testing Support Copilot...")settings = get_settings()try:    copilot = SupportCopilot(settings=settings)    print("✅ Support Copilot initialized successfully!")        # Test draft generation    test_ticket = {        "id": "TICKET-001",        "subject": "Cannot reset password",        "description": "I've been trying to reset my password for the account but keep getting an error.",        "priority": "high",        "status": "open"    }        test_customer = {        "id": "CUST-001",        "name": "John Doe",        "email": "john@example.com",        "company": "Acme Corp"    }        result = copilot.generate_draft(test_ticket, test_customer)    print(f"✅ Draft generated successfully!")    print(f"   Length: {len(result['draft'])} characters")    print(f"   Memory hits: {result['context_used']['signals']['memory_hit_count']}")    print(f"   Knowledge hits: {result['context_used']['signals']['knowledge_hit_count']}")except Exception as e:    print(f"⚠️  Error in copilot: {e}")    import traceback    traceback.print_exc()print("\n✅ Core Services Phase Complete!")

## PHASE 5️⃣: Data Models & Schemas### 📊 Pydantic Schemas (LangChain 1.2.0+ compatible)Request/response schemas for API endpoints.

In [ ]:
# PHASE 5: Data Models & Schemas (Pydantic 2.0+ compatible)from pydantic import BaseModel, EmailStr, Fieldfrom typing import Optional, Anyfrom datetime import datetime# ==================== Ticket Models ====================class TicketBase(BaseModel):    """Base ticket model."""    subject: str = Field(..., min_length=3, max_length=500)    description: str = Field(..., min_length=5, max_length=5000)    priority: str = Field(default="normal", pattern="^(low|normal|high|critical)$")    status: str = Field(default="open", pattern="^(open|in_progress|resolved|closed)$")class TicketCreate(TicketBase):    """Create ticket request."""    customer_id: str = Field(..., description="Customer ID")class TicketResponse(TicketBase):    """Ticket response model."""    id: str    customer_id: str    created_at: datetime        class Config:        from_attributes = True# ==================== Customer Models ====================class CustomerBase(BaseModel):    """Base customer model."""    name: str = Field(..., min_length=2, max_length=100)    email: EmailStr    company: Optional[str] = Field(None, max_length=100)class CustomerCreate(CustomerBase):    """Create customer request."""    passclass CustomerResponse(CustomerBase):    """Customer response model."""    id: str    created_at: datetime        class Config:        from_attributes = True# ==================== Draft Models ====================class DraftRequest(BaseModel):    """Draft generation request."""    ticket_id: str    customer_id: str    ticket_subject: str = Field(..., min_length=3)    ticket_description: str = Field(..., min_length=5)    customer_name: str = Field(..., min_length=2)    customer_email: EmailStr    customer_company: Optional[str] = Noneclass ContextSignals(BaseModel):    """Context signals metadata."""    memory_hit_count: int    knowledge_hit_count: int    tool_call_count: int    knowledge_sources: list[str]class DraftResponse(BaseModel):    """Draft generation response."""    draft: str    context_used: dict[str, Any]# ==================== Memory Models ====================class MemoryQuery(BaseModel):    """Memory search query."""    customer_email: EmailStr    query: str = Field(..., min_length=3, max_length=500)    customer_company: Optional[str] = None    limit: int = Field(default=10, le=50)class MemoryItem(BaseModel):    """Memory item."""    id: str    memory: str    metadata: dict[str, Any]    distance: Optional[float] = Noneclass MemoryListResponse(BaseModel):    """Memory list response."""    items: list[MemoryItem]    count: int# ==================== Health Models ====================class HealthResponse(BaseModel):    """Health check response."""    status: str    timestamp: datetime    version: str    components: dict[str, str]# Test schemasprint("\n📊 Testing Data Models...")try:    # Test creating customer    customer = CustomerCreate(        name="Jane Doe",        email="jane@example.com",        company="TechCorp"    )    print(f"✅ Customer model created: {customer.name}")        # Test creating ticket    ticket = TicketCreate(        subject="API integration issue",        description="Getting 403 error when calling the API",        customer_id="CUST-001",        priority="high"    )    print(f"✅ Ticket model created: {ticket.subject}")        # Test draft response    draft = DraftResponse(        draft="Dear Jane, we are investigating your issue...",        context_used={"version": 2, "signals": {}}    )    print(f"✅ Draft response model created")    except Exception as e:    print(f"⚠️  Error in models: {e}")print("\n✅ Schemas Phase Complete!")

## PHASE 6️⃣: Database Setup (SQLite)### 🗄️ Customer & Ticket DatabaseSQLite database for persisting customer and ticket data.

In [ ]:
# PHASE 6: Database Setup (SQLAlchemy + SQLite)from sqlalchemy import create_engine, Column, String, DateTime, Integer, Textfrom sqlalchemy.ext.declarative import declarative_basefrom sqlalchemy.orm import sessionmaker, Sessionfrom datetime import datetimefrom pathlib import Pathimport uuidBase = declarative_base()# ==================== Database Models ====================class CustomerModel(Base):    """Customer database model."""    __tablename__ = "customers"        id = Column(String(36), primary_key=True, default=lambda: str(uuid.uuid4()))    name = Column(String(100), nullable=False)    email = Column(String(100), unique=True, nullable=False, index=True)    company = Column(String(100), nullable=True)    created_at = Column(DateTime, default=datetime.utcnow)class TicketModel(Base):    """Ticket database model."""    __tablename__ = "tickets"        id = Column(String(36), primary_key=True, default=lambda: str(uuid.uuid4()))    customer_id = Column(String(36), nullable=False, index=True)    subject = Column(String(500), nullable=False)    description = Column(Text, nullable=False)    priority = Column(String(20), default="normal")    status = Column(String(20), default="open")    created_at = Column(DateTime, default=datetime.utcnow)class DraftModel(Base):    """Draft response database model."""    __tablename__ = "drafts"        id = Column(String(36), primary_key=True, default=lambda: str(uuid.uuid4()))    ticket_id = Column(String(36), nullable=False, index=True)    customer_id = Column(String(36), nullable=False)    draft_content = Column(Text, nullable=False)    context_used = Column(Text, nullable=True)  # JSON string    accepted = Column(Integer, default=0)  # Boolean as integer    created_at = Column(DateTime, default=datetime.utcnow)# ==================== Database Manager ====================class DatabaseManager:    """Manages database connections and operations."""        def __init__(self, settings: Settings):        """Initialize database manager."""        self.settings = settings                # Ensure data directory exists        self.settings.resolve(self.settings.data_dir).mkdir(parents=True, exist_ok=True)                # Create database engine        db_url = f"sqlite:///{self.settings.db_file}"        self.engine = create_engine(db_url, echo=False)        self.SessionLocal = sessionmaker(bind=self.engine)                # Create tables        Base.metadata.create_all(self.engine)        print(f"✅ Database initialized at: {self.settings.db_file}")        def get_session(self) -> Session:        """Get database session."""        return self.SessionLocal()        def add_customer(self, name: str, email: str, company: Optional[str] = None) -> CustomerModel:        """Add customer to database."""        session = self.get_session()        try:            customer = CustomerModel(name=name, email=email, company=company)            session.add(customer)            session.commit()            return customer        except Exception as e:            session.rollback()            raise e        finally:            session.close()        def get_customer(self, email: str) -> Optional[CustomerModel]:        """Get customer by email."""        session = self.get_session()        try:            return session.query(CustomerModel).filter(CustomerModel.email == email).first()        finally:            session.close()        def add_ticket(        self,        customer_id: str,        subject: str,        description: str,        priority: str = "normal"    ) -> TicketModel:        """Add ticket to database."""        session = self.get_session()        try:            ticket = TicketModel(                customer_id=customer_id,                subject=subject,                description=description,                priority=priority            )            session.add(ticket)            session.commit()            return ticket        except Exception as e:            session.rollback()            raise e        finally:            session.close()        def get_ticket(self, ticket_id: str) -> Optional[TicketModel]:        """Get ticket by ID."""        session = self.get_session()        try:            return session.query(TicketModel).filter(TicketModel.id == ticket_id).first()        finally:            session.close()        def save_draft(        self,        ticket_id: str,        customer_id: str,        draft_content: str,        context_used: Optional[str] = None    ) -> DraftModel:        """Save generated draft."""        session = self.get_session()        try:            draft = DraftModel(                ticket_id=ticket_id,                customer_id=customer_id,                draft_content=draft_content,                context_used=context_used            )            session.add(draft)            session.commit()            return draft        except Exception as e:            session.rollback()            raise e        finally:            session.close()# Test database setupprint("\n🗄️ Testing Database Setup...")settings = get_settings()try:    db_manager = DatabaseManager(settings=settings)        # Add test customer    customer = db_manager.add_customer(        name="Test Customer",        email=f"test_{uuid.uuid4().hex[:8]}@example.com",        company="Test Corp"    )    print(f"✅ Test customer added: {customer.name} (ID: {customer.id})")        # Add test ticket    ticket = db_manager.add_ticket(        customer_id=customer.id,        subject="Test issue",        description="This is a test ticket",        priority="high"    )    print(f"✅ Test ticket added: {ticket.subject} (ID: {ticket.id})")        # Retrieve customer    retrieved = db_manager.get_customer(customer.email)    print(f"✅ Retrieved customer: {retrieved.name}")    except Exception as e:    print(f"⚠️  Error in database: {e}")    import traceback    traceback.print_exc()print("\n✅ Database Phase Complete!")

## PHASE 7️⃣: API Routes & Endpoints### 🔌 FastAPI Routes (LangChain 1.2.0+ compatible)RESTful API endpoints for copilot functionality.

In [ ]:
# PHASE 7: API Routes (FastAPI - LangChain 1.2.0+ compatible)from fastapi import APIRouter, Depends, HTTPException, statusfrom typing import Optional, Anyimport jsonfrom datetime import datetime# Create routerscopilot_router = APIRouter(prefix="/api/v1/copilot", tags=["Copilot"])knowledge_router = APIRouter(prefix="/api/v1/knowledge", tags=["Knowledge Base"])memory_router = APIRouter(prefix="/api/v1/memory", tags=["Memory"])health_router = APIRouter(prefix="/api/v1/health", tags=["Health"])# Global dependenciesdef get_settings() -> Settings:    """Dependency: Get settings."""    return get_settings()def get_copilot(settings: Settings = Depends(get_settings)) -> SupportCopilot:    """Dependency: Get copilot instance."""    if not hasattr(get_copilot, '_instance'):        get_copilot._instance = SupportCopilot(settings=settings)    return get_copilot._instancedef get_db_manager(settings: Settings = Depends(get_settings)) -> DatabaseManager:    """Dependency: Get database manager."""    if not hasattr(get_db_manager, '_instance'):        get_db_manager._instance = DatabaseManager(settings=settings)    return get_db_manager._instance# ==================== COPILOT ENDPOINTS ====================@copilot_router.post(    "/generate-draft",    response_model=DraftResponse,    summary="Generate support response draft",    description="Use AI to generate a support response draft for a customer ticket")async def generate_draft(    request: DraftRequest,    copilot: SupportCopilot = Depends(get_copilot),    db: DatabaseManager = Depends(get_db_manager),) -> DraftResponse:    """    Generate a support response draft using AI.        - Uses customer info and ticket details    - Searches memory for past resolutions    - Queries knowledge base for relevant docs    - Calls tools to gather customer context    - Returns draft with context metadata    """    try:        # Prepare ticket data        ticket = {            "id": request.ticket_id,            "subject": request.ticket_subject,            "description": request.ticket_description,            "priority": "normal",            "status": "open",        }                # Prepare customer data        customer = {            "id": request.customer_id,            "name": request.customer_name,            "email": request.customer_email,            "company": request.customer_company,        }                # Generate draft        result = copilot.generate_draft(ticket=ticket, customer=customer)                # Save draft to database        draft_obj = db.save_draft(            ticket_id=request.ticket_id,            customer_id=request.customer_id,            draft_content=result["draft"],            context_used=json.dumps(result["context_used"])        )                return DraftResponse(            draft=result["draft"],            context_used=result["context_used"]        )        except Exception as e:        raise HTTPException(            status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,            detail=f"Failed to generate draft: {str(e)}"        )@copilot_router.get(    "/tickets/{ticket_id}",    response_model=TicketResponse,    summary="Get ticket details")async def get_ticket(    ticket_id: str,    db: DatabaseManager = Depends(get_db_manager),) -> TicketResponse:    """Retrieve ticket details."""    ticket = db.get_ticket(ticket_id)    if not ticket:        raise HTTPException(            status_code=status.HTTP_404_NOT_FOUND,            detail="Ticket not found"        )    return TicketResponse(**ticket.__dict__)# ==================== KNOWLEDGE BASE ENDPOINTS ====================@knowledge_router.get(    "/search",    summary="Search knowledge base",    description="Search knowledge base using semantic similarity")async def search_knowledge(    q: str,    top_k: int = 4,    copilot: SupportCopilot = Depends(get_copilot),) -> dict[str, Any]:    """Search knowledge base for relevant documents."""    if not q or len(q) < 3:        raise HTTPException(            status_code=status.HTTP_400_BAD_REQUEST,            detail="Query must be at least 3 characters"        )        results = copilot.rag.search(query=q, top_k=top_k)        return {        "query": q,        "results": results,        "count": len(results),        "timestamp": datetime.utcnow().isoformat()    }@knowledge_router.post(    "/load",    summary="Load knowledge base documents",    description="Load markdown documents from knowledge base folder")async def load_knowledge(    copilot: SupportCopilot = Depends(get_copilot),) -> dict[str, Any]:    """Load documents from knowledge base folder."""    doc_count = copilot.rag.load_documents_from_folder()        return {        "status": "success",        "documents_loaded": doc_count,        "timestamp": datetime.utcnow().isoformat()    }# ==================== MEMORY ENDPOINTS ====================@memory_router.post(    "/search",    summary="Search customer memories",    description="Search past resolutions and interactions for a customer")async def search_memory(    query: MemoryQuery,    copilot: SupportCopilot = Depends(get_copilot),) -> MemoryListResponse:    """Search customer memory system."""    if not copilot.memory:        raise HTTPException(            status_code=status.HTTP_503_SERVICE_UNAVAILABLE,            detail="Memory system is not available"        )        memories = copilot.memory.search(        query=query.query,        user_id=query.customer_email,        limit=query.limit    )        return MemoryListResponse(        items=memories,        count=len(memories)    )@memory_router.get(    "/customer/{customer_email}",    summary="Get customer memories",    description="List all memories for a customer")async def list_customer_memories(    customer_email: str,    limit: int = 20,    copilot: SupportCopilot = Depends(get_copilot),) -> MemoryListResponse:    """List customer memories."""    if not copilot.memory:        raise HTTPException(            status_code=status.HTTP_503_SERVICE_UNAVAILABLE,            detail="Memory system is not available"        )        memories = copilot.memory.list_memories(        user_id=customer_email,        limit=limit    )        return MemoryListResponse(        items=memories,        count=len(memories)    )# ==================== HEALTH ENDPOINTS ====================@health_router.get(    "/status",    response_model=HealthResponse,    summary="Health check")async def health_check(    settings: Settings = Depends(get_settings),    copilot: SupportCopilot = Depends(get_copilot),) -> HealthResponse:    """Check application health status."""    components = {        "llm": "ok" if settings.groq_api_key else "error",        "rag": "ok",        "memory": "ok" if copilot.memory else "unavailable",    }        return HealthResponse(        status="healthy" if all(v == "ok" for v in components.values()) else "degraded",        timestamp=datetime.utcnow(),        version="1.0.0",        components=components    )# ==================== VERSION ENDPOINT ====================@health_router.get(    "/version",    summary="Get API version")async def get_version(    settings: Settings = Depends(get_settings),) -> dict[str, Any]:    """Get API and dependency versions."""    import langchain    import langgraph        return {        "app_version": settings.app_version,        "langchain_version": langchain.__version__,        "langgraph_version": langgraph.__version__,        "python_version": "3.9+",        "timestamp": datetime.utcnow().isoformat()    }print("\n🔌 API Routes configured:")print("   ✅ Copilot routes: /api/v1/copilot")print("   ✅ Knowledge routes: /api/v1/knowledge")print("   ✅ Memory routes: /api/v1/memory")print("   ✅ Health routes: /api/v1/health")print("\n✅ API Routes Phase Complete!")

## PHASE 8️⃣: FastAPI Application Factory### 🏭 Create FastAPI ApplicationInitialize and configure the FastAPI app with CORS, middleware, and routes.

In [ ]:
# PHASE 8: FastAPI Application Factory (LangChain 1.2.0+ compatible)from fastapi import FastAPIfrom fastapi.middleware.cors import CORSMiddlewarefrom fastapi.responses import JSONResponseimport timedef create_app(settings: Optional[Settings] = None) -> FastAPI:    """    Create and configure FastAPI application.        Args:        settings: Optional Settings instance        Returns:        Configured FastAPI application    """    if settings is None:        settings = get_settings()        # Ensure directories exist    ensure_directories(settings)        # Create FastAPI app    app = FastAPI(        title=settings.app_name,        description="AI-powered customer support agent with memory and tool calling",        version=settings.app_version,        docs_url="/api/v1/docs",        redoc_url="/api/v1/redoc",        openapi_url="/api/v1/openapi.json",    )        # ==================== CORS Middleware ====================    app.add_middleware(        CORSMiddleware,        allow_origins=["*"],  # In production: specify allowed origins        allow_credentials=True,        allow_methods=["*"],        allow_headers=["*"],    )        # ==================== Custom Middleware ====================    @app.middleware("http")    async def add_process_time_header(request, call_next):        """Add processing time header to response."""        start_time = time.time()        response = await call_next(request)        process_time = time.time() - start_time        response.headers["X-Process-Time"] = str(process_time)        return response        # ==================== Exception Handlers ====================    @app.exception_handler(ValueError)    async def value_error_handler(request, exc):        """Handle ValueError exceptions."""        return JSONResponse(            status_code=400,            content={"detail": str(exc)},        )        @app.exception_handler(Exception)    async def general_exception_handler(request, exc):        """Handle general exceptions."""        return JSONResponse(            status_code=500,            content={"detail": "Internal server error"},        )        # ==================== Root Endpoint ====================    @app.get("/")    async def root():        """Root endpoint."""        return {            "message": "AI Customer Support Agent API",            "version": settings.app_version,            "docs": "/api/v1/docs",            "health": "/api/v1/health/status"        }        @app.get("/api")    async def api_root():        """API root endpoint."""        return {            "message": "Customer Support Agent API v1",            "endpoints": {                "copilot": "/api/v1/copilot/generate-draft",                "knowledge": "/api/v1/knowledge/search",                "memory": "/api/v1/memory/search",                "health": "/api/v1/health/status",            }        }        # ==================== Register Routers ====================    app.include_router(copilot_router)    app.include_router(knowledge_router)    app.include_router(memory_router)    app.include_router(health_router)        return app# Create app instanceprint("\n🏭 Creating FastAPI Application...")settings = get_settings()try:    app = create_app(settings=settings)    print("✅ FastAPI application created successfully!")    print(f"   App Name: {settings.app_name}")    print(f"   Version: {settings.app_version}")    print(f"   API Docs: /api/v1/docs")except Exception as e:    print(f"⚠️  Error creating app: {e}")    import traceback    traceback.print_exc()print("\n✅ FastAPI Application Factory Phase Complete!")

## PHASE 9️⃣: Example Usage & Testing### 🧪 Complete Example: Generate Support DraftEnd-to-end example showing how to use the system.

In [ ]:
# PHASE 9: Complete Example - Generate Support Draftimport jsonfrom pprint import pprintprint("\n" + "="*70)print("COMPLETE EXAMPLE: AI CUSTOMER SUPPORT AGENT")print("="*70)# Initialize all componentssettings = get_settings()db_manager = DatabaseManager(settings=settings)copilot = SupportCopilot(settings=settings)# ==================== STEP 1: Create Customer ====================print("\n[STEP 1] Creating Customer...")print("-" * 70)customer_data = {    "name": "Alice Johnson",    "email": f"alice_{int(datetime.now().timestamp())}@techcorp.com",    "company": "TechCorp Inc"}try:    customer = db_manager.add_customer(**customer_data)    print(f"✅ Customer created:")    print(f"   ID: {customer.id}")    print(f"   Name: {customer.name}")    print(f"   Email: {customer.email}")    print(f"   Company: {customer.company}")except Exception as e:    print(f"⚠️  Error creating customer: {e}")    customer = type('obj', (object,), {        'id': 'CUST-001',        'name': customer_data['name'],        'email': customer_data['email'],        'company': customer_data['company']    })()# ==================== STEP 2: Create Ticket ====================print("\n[STEP 2] Creating Support Ticket...")print("-" * 70)ticket_data = {    "customer_id": customer.id,    "subject": "Unable to export reports to CSV",    "description": "When I click the export button on the reports page, I get an error. The system shows 'Export failed (code 502)'. This is blocking our monthly reporting workflow.",    "priority": "high"}try:    ticket = db_manager.add_ticket(**ticket_data)    print(f"✅ Ticket created:")    print(f"   ID: {ticket.id}")    print(f"   Subject: {ticket.subject}")    print(f"   Priority: {ticket.priority}")    print(f"   Status: {ticket.status}")except Exception as e:    print(f"⚠️  Error creating ticket: {e}")    ticket = type('obj', (object,), {        'id': 'TICKET-001',        'subject': ticket_data['subject'],        'description': ticket_data['description'],        'priority': ticket_data['priority'],        'status': 'open'    })()# ==================== STEP 3: Add Sample Knowledge ====================print("\n[STEP 3] Loading Knowledge Base...")print("-" * 70)sample_knowledge = [    {        "doc_id": "kb_export_csv",        "content": "CSV Export Troubleshooting: If you encounter a 502 error during export, it's usually due to temporary server issues. Try: 1) Refresh the page and retry, 2) Reduce the report date range, 3) Contact support if issue persists",        "source": "export_guide.md"    },    {        "doc_id": "kb_error_502",        "content": "Error 502 Bad Gateway: This indicates the server couldn't process your request. Common causes: server overload, network timeout, or file size exceeding limits. Solutions: wait a few minutes and retry, or contact our technical team",        "source": "error_codes.md"    },]try:    for kb_item in sample_knowledge:        copilot.rag.add_knowledge(**kb_item)    print(f"✅ Knowledge base loaded: {len(sample_knowledge)} documents")except Exception as e:    print(f"⚠️  Error loading knowledge: {e}")# ==================== STEP 4: Add Sample Memory ====================print("\n[STEP 4] Adding Customer Memory...")print("-" * 70)sample_memory = {    "user_id": customer.email.lower(),    "ticket_subject": "Previous export issue",    "ticket_description": "Customer had trouble exporting data last month",    "accepted_draft": "We resolved the export issue by clearing browser cache and updating the browser. Customer was satisfied.",    "entity_links": ["export_issue", "browser_cache"]}try:    if copilot.memory:        copilot.memory.add_resolution(**sample_memory)        print(f"✅ Memory added for: {customer.email}")    else:        print("⚠️  Memory system not available")except Exception as e:    print(f"⚠️  Error adding memory: {e}")# ==================== STEP 5: Generate Support Draft ====================print("\n[STEP 5] Generating AI Support Draft...")print("-" * 70)print("This will use:")print("  • Tool calling (lookup customer plan, billing status, etc)")print("  • Memory retrieval (past resolutions)")print("  • Knowledge base search (relevant docs)")print("  • LLM generation (Groq + LangChain 1.2.0+)")print()draft_input = {    "ticket": {        "id": ticket.id,        "subject": ticket.subject,        "description": ticket.description,        "priority": ticket.priority,        "status": ticket.status    },    "customer": {        "id": customer.id,        "name": customer.name,        "email": customer.email,        "company": customer.company    }}try:    print("🔄 Generating draft (this may take 10-30 seconds)...")    result = copilot.generate_draft(**draft_input)        print("\n✅ DRAFT GENERATED SUCCESSFULLY!")    print("\n" + "="*70)    print("GENERATED RESPONSE:")    print("="*70)    print(result["draft"])        # Save draft to database    draft_saved = db_manager.save_draft(        ticket_id=ticket.id,        customer_id=customer.id,        draft_content=result["draft"],        context_used=json.dumps(result["context_used"])    )    print(f"\n✅ Draft saved to database (ID: {draft_saved.id})")        # Display context metadata    print("\n" + "="*70)    print("CONTEXT & SIGNALS:")    print("="*70)    context = result["context_used"]    print(f"\nVersion: {context.get('version', 'N/A')}")    print(f"Agent Runtime: {context.get('agent_runtime', 'N/A')}")        signals = context.get("signals", {})    print(f"\nSignals:")    print(f"  • Memory hits: {signals.get('memory_hit_count', 0)}")    print(f"  • Knowledge hits: {signals.get('knowledge_hit_count', 0)}")    print(f"  • Tool calls: {signals.get('tool_call_count', 0)}")        if signals.get('knowledge_sources'):        print(f"  • Knowledge sources: {', '.join(signals['knowledge_sources'])}")        errors = context.get("errors", [])    if errors:        print(f"\nWarnings:")        for error in errors:            print(f"  ⚠️  {error}")    except Exception as e:    print(f"⚠️  Error generating draft: {e}")    print(f"\nFallback response (using template):")    fallback = f"""Dear {customer.name},Thank you for contacting our support team regarding the CSV export issue.We understand how critical this is for your monthly reporting workflow. Our team is investigating the 502 error you encountered on the export function.Based on our systems, here are the immediate steps you can try:1. **Clear Your Browser Cache**: Sometimes cached files can interfere with the export process2. **Reduce the Date Range**: If exporting a large dataset, try splitting it into smaller date ranges3. **Try a Different Browser**: This helps us identify if it's browser-specificIf these steps don't resolve the issue, please reply with:- The exact time the error occurred- The date range you were trying to export- Your browser and versionOur technical team will investigate further and provide a solution within 24 hours.Best regards,Customer Support Team"""    print(fallback)print("\n" + "="*70)print("✅ EXAMPLE COMPLETE")print("="*70)

## PHASE 🔟: Deployment Guide### 📋 Step-by-Step Deployment InstructionsThis section provides detailed deployment instructions for different environments.### 📌 TABLE OF CONTENTS1. **Local Development Setup**2. **Google Colab Deployment**3. **Production Deployment (Heroku/Railway)**4. **Docker Deployment**5. **API Testing & Verification**---## 1️⃣ LOCAL DEVELOPMENT SETUP### Prerequisites- Python 3.9+- pip or conda- Git- Text editor or IDE### Installation Steps```bash# Clone the repositorygit clone https://github.com/yourusername/customer-support-agent.gitcd customer_support_agent# Create virtual environmentpython -m venv venvsource venv/bin/activate  # On Windows: venv\Scripts\activate# Install dependencies (LangChain 1.2.0+)pip install -r requirements.txt# Set up environment variablescp .env.example .env# Edit .env with your API keys:# - GROQ_API_KEY=your_key_here# - GOOGLE_API_KEY=optional# - TAVILY_API_KEY=optional# Create necessary directoriesmkdir -p data/chroma_rag data/chroma_mem0 knowledge_base# Run the applicationpython -m customer_support_agent.main```The application will start at `http://localhost:8000`- API Docs: http://localhost:8000/api/v1/docs- ReDoc: http://localhost:8000/api/v1/redoc---## 2️⃣ GOOGLE COLAB DEPLOYMENT### Setup Steps (Already Completed in This Notebook!)1. ✅ **Dependencies Installed** (Phase 1)2. ✅ **Configuration Set** (Phase 2)3. ✅ **Services Initialized** (Phases 3-4)4. ✅ **Database Created** (Phase 6)5. ✅ **API Routes Ready** (Phase 7-8)### Running the API in Colab```python# Use ngrok to expose the API to the internet!pip install -q pyngrokfrom pyngrok import ngrok# Authenticate ngrok (get token from https://dashboard.ngrok.com)ngrok.set_auth_token("your_ngrok_token")# Start ngrok tunnelpublic_url = ngrok.connect(8000)print(f"Public API URL: {public_url}")# Run FastAPI appimport uvicornuvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")```### Accessing the APIAfter starting ngrok, your API will be available at:- **Base URL**: `{public_url}/api/v1`- **Docs**: `{public_url}/api/v1/docs`### Example API Call```bashcurl -X POST "https://your-ngrok-url/api/v1/copilot/generate-draft" \  -H "Content-Type: application/json" \  -d '{    "ticket_id": "TICKET-001",    "customer_id": "CUST-001",    "ticket_subject": "Cannot reset password",    "ticket_description": "I am unable to reset my account password",    "customer_name": "John Doe",    "customer_email": "john@example.com",    "customer_company": "Acme Corp"  }'```---## 3️⃣ PRODUCTION DEPLOYMENT (Heroku/Railway)### Option A: Heroku Deployment#### Prerequisites- Heroku account (https://www.heroku.com)- Heroku CLI installed- GitHub account#### Steps```bash# Login to Herokuheroku login# Create Heroku appheroku create your-app-name# Add buildpack for Pythonheroku buildpacks:add heroku/python# Set environment variablesheroku config:set GROQ_API_KEY=your_keyheroku config:set GOOGLE_API_KEY=your_key (optional)# Create Procfile in project rootecho "web: uvicorn customer_support_agent.main:app --host 0.0.0.0 --port \$PORT" > Procfile# Create runtime.txt for Python versionecho "python-3.11.7" > runtime.txt# Push to Herokugit push heroku main# View logsheroku logs --tail```#### Option B: Railway Deployment```bash# Install Railway CLInpm install -g @railway/cli# Loginrailway login# Link projectrailway link# Set environment variablesrailway variable:set GROQ_API_KEY=your_key# Deployrailway up# Get URLrailway domain```### Production Checklist- [ ] Environment variables configured- [ ] Database migrations run- [ ] Logging configured- [ ] Rate limiting enabled- [ ] CORS properly configured- [ ] API keys secured (use environment variables)- [ ] SSL/HTTPS enabled- [ ] Monitoring set up- [ ] Backup strategy for vector DB- [ ] API documentation updated---## 4️⃣ DOCKER DEPLOYMENT### DockerfileCreate `Dockerfile` in project root:```dockerfileFROM python:3.11-slimWORKDIR /app# Install system dependenciesRUN apt-get update && apt-get install -y \    build-essential \    curl \    && rm -rf /var/lib/apt/lists/*# Copy requirementsCOPY requirements.txt .# Install Python dependenciesRUN pip install --no-cache-dir -r requirements.txt# Copy applicationCOPY . .# Create necessary directoriesRUN mkdir -p data/chroma_rag data/chroma_mem0 knowledge_base# Expose portEXPOSE 8000# Health checkHEALTHCHECK --interval=30s --timeout=10s --start-period=5s --retries=3 \    CMD curl -f http://localhost:8000/api/v1/health/status || exit 1# Run applicationCMD ["uvicorn", "customer_support_agent.main:app", "--host", "0.0.0.0", "--port", "8000"]```### Docker ComposeCreate `docker-compose.yml`:```yamlversion: '3.8'services:  api:    build: .    ports:      - "8000:8000"    environment:      - GROQ_API_KEY=${GROQ_API_KEY}      - GOOGLE_API_KEY=${GOOGLE_API_KEY}      - TAVILY_API_KEY=${TAVILY_API_KEY}    volumes:      - ./data:/app/data      - ./knowledge_base:/app/knowledge_base    restart: unless-stopped  chroma:    image: chromadb/chroma:latest    ports:      - "8001:8000"    volumes:      - chroma_data:/chroma/data    restart: unless-stoppedvolumes:  chroma_data:```### Building and Running```bash# Build imagedocker build -t customer-support-agent:latest .# Run containerdocker run -p 8000:8000 \  -e GROQ_API_KEY=your_key \  customer-support-agent:latest# Or use Docker Composedocker-compose up -d```---## 5️⃣ API TESTING & VERIFICATION### Health Check```bashcurl http://localhost:8000/api/v1/health/status```Expected response:```json{  "status": "healthy",  "version": "1.0.0",  "components": {    "llm": "ok",    "rag": "ok",    "memory": "ok"  }}```### Generate Draft Test```bashcurl -X POST http://localhost:8000/api/v1/copilot/generate-draft \  -H "Content-Type: application/json" \  -d '{    "ticket_id": "TICKET-001",    "customer_id": "CUST-001",    "ticket_subject": "API Integration Help",    "ticket_description": "Having trouble integrating the API",    "customer_name": "Jane Doe",    "customer_email": "jane@example.com",    "customer_company": "TechCorp"  }'```### Knowledge Base Search```bashcurl "http://localhost:8000/api/v1/knowledge/search?q=how%20to%20reset%20password"```### Memory Search```bashcurl -X POST http://localhost:8000/api/v1/memory/search \  -H "Content-Type: application/json" \  -d '{    "customer_email": "jane@example.com",    "query": "password reset",    "limit": 5  }'```---## 📚 REQUIREMENTS.TXT```# Core dependencieslangchain>=1.2.0langchain-core>=0.2.0langchain-community>=0.2.0langgraph>=0.1.0langchain-groq>=0.1.0# LLM & APIsgroq>=0.7.0google-genai>=0.3.0# Vector Databasechromadb>=0.4.0mem0ai>=0.1.0# Web Frameworkfastapi>=0.104.0uvicorn[standard]>=0.24.0pydantic>=2.0pydantic-settings>=2.0# Databasesqlalchemy>=2.0# Utilitiespython-dotenv>=1.0.0requests>=2.31.0pyngrok>=5.2.0# Testingpytest>=7.4.0pytest-asyncio>=0.21.0httpx>=0.24.0```---## 🔧 TROUBLESHOOTING### Issue: GROQ_API_KEY not found**Solution**: Set the environment variable in Colab Secrets (click 🔑 icon)### Issue: ChromaDB connection failed**Solution**: Ensure `data/chroma_*` directories are writable```bashchmod -R 755 data/```### Issue: Memory system unavailable**Solution**: Check ChromaDB installation and file permissions```pythonimport chromadbprint(chromadb.__version__)```### Issue: LLM generating slow responses**Solution**: Use `llama-3.1-8b-instant` instead of `70b-versatile````pythonsettings.groq_model = "llama-3.1-8b-instant"```### Issue: Token limit exceeded**Solution**: Reduce `llm_max_tokens` and `rag_top_k` in settings```pythonsettings.llm_max_tokens = 1024settings.rag_top_k = 2```---## 📊 MONITORING & LOGGING### Enable Debug Logging```pythonimport logginglogging.basicConfig(level=logging.DEBUG)```### Monitor API Performance```bash# View request logstail -f logs/api.log# Monitor resource usagewatch -n 1 'ps aux | grep python'```### Set Up AlertsFor production, use services like:- **Sentry** (error tracking)- **DataDog** (monitoring)- **New Relic** (APM)---## 🚀 NEXT STEPS1. **Customize Support Tools**: Add company-specific tools in `integrations/tools/`2. **Expand Knowledge Base**: Add more markdown documents to `knowledge_base/`3. **Fine-tune Prompts**: Adjust system prompts in `services/copilot_service.py`4. **Add Authentication**: Implement API key or OAuth2 authentication5. **Set Up Webhooks**: Integrate with ticketing system (Zendesk, Intercom, etc.)---

## PHASE 1️⃣1️⃣: Summary & Important Notes### ✅ WHAT YOU'VE LEARNEDThis notebook demonstrates a **complete, production-ready AI customer support system** using:1. **LangChain 1.2.0+** with latest imports and best practices2. **LangGraph** for agent orchestration with checkpointing3. **Groq API** for fast LLM inference (free tier available)4. **ChromaDB** for vector storage (Memory + RAG)5. **FastAPI** for REST API endpoints6. **SQLAlchemy** for persistent database### 🎯 KEY FEATURES IMPLEMENTED| Feature | Implementation | Status ||---------|-----------------|--------|| Tool Calling | LangChain 1.2.0+ @tool decorator | ✅ Complete || Memory System | Mem0 + ChromaDB persistent store | ✅ Complete || RAG System | ChromaDB vector search | ✅ Complete || LLM Integration | Groq + LangChain ChatGroq | ✅ Complete || Agent Orchestration | LangGraph with InMemorySaver | ✅ Complete || API Endpoints | FastAPI with dependency injection | ✅ Complete || Database | SQLAlchemy + SQLite | ✅ Complete || Deployment | Docker, Heroku, Railway ready | ✅ Complete |### 🔑 API KEYS REQUIRED| Service | Purpose | Cost | Get Key ||---------|---------|------|---------|| **GROQ_API_KEY** | LLM inference | Free | https://console.groq.com || GOOGLE_API_KEY | Embeddings (optional) | Free | https://ai.google.dev || TAVILY_API_KEY | Web search (optional) | Free | https://tavily.com |### 📦 VERSION SPECIFICATIONS```✅ CONFIRMED VERSIONS FOR THIS NOTEBOOK:- LangChain: >=1.2.0- LangChain-Core: >=0.2.0- LangChain-Community: >=0.2.0- LangGraph: >=0.1.0- LangChain-Groq: >=0.1.0- Groq: >=0.7.0- ChromaDB: >=0.4.0- FastAPI: >=0.104.0- Pydantic: >=2.0```### 🏗️ PROJECT STRUCTURE```customer_support_agent/├── 📄 PHASE 1: Environment Setup├── 📄 PHASE 2: Configuration (Settings)├── 📄 PHASE 3: Integrations (Tools, Memory, RAG)├── 📄 PHASE 4: Core Services (Copilot)├── 📄 PHASE 5: Data Models (Pydantic schemas)├── 📄 PHASE 6: Database (SQLAlchemy)├── 📄 PHASE 7: API Routes (FastAPI)├── 📄 PHASE 8: Application Factory├── 📄 PHASE 9: Example Usage & Testing└── 📄 PHASE 10: Deployment Guide (Step-by-step)```### 💡 IMPORTANT NOTES1. **Token Limits**:    - Default `llm_max_tokens=2048`    - Groq free tier: 6000 TPM   - Reduce if hitting limits: `settings.llm_max_tokens = 1024`2. **ChromaDB Storage**:   - Persistent at `data/chroma_rag/` and `data/chroma_mem0/`   - In Colab, use Google Drive at `/content/drive/MyDrive/`   - Survives notebook restarts when saved to Drive3. **Memory Scope**:   - **Customer scope**: Individual customer email   - **Company scope**: All customers from same company   - Normalized as `company::company-name`4. **RAG Best Practices**:   - Chunk size: 800 tokens (default)   - Overlap: 120 tokens (prevents content loss)   - Top K: 4 documents (balance relevance vs. tokens)5. **Error Handling**:   - Memory failures don't block draft generation   - Falls back to LLM-only if agent fails   - Last resort: Deterministic template response6. **Production Checklist**:   - [ ] Set specific CORS origins   - [ ] Add authentication (API keys/OAuth2)   - [ ] Enable rate limiting   - [ ] Configure logging/monitoring   - [ ] Set up automated backups   - [ ] Test failover scenarios### 🎓 LEARNING OUTCOMESAfter completing this notebook, you can:1. ✅ Build LangChain 1.2.0+ applications from scratch2. ✅ Implement multi-agent systems with LangGraph3. ✅ Design memory and RAG systems4. ✅ Create FastAPI applications with dependency injection5. ✅ Deploy AI applications to production6. ✅ Handle tool calling and function execution7. ✅ Manage prompts and context windows efficiently### 🚀 NEXT STEPS FOR CUSTOMIZATION1. **Add Custom Tools**:   ```python   @tool(name="check_user_subscription")   def check_user_subscription(user_id: str) -> str:       # Your implementation       pass   ```2. **Expand Knowledge Base**:   - Add `.md` files to `knowledge_base/` folder   - Call `copilot.rag.load_documents_from_folder()`3. **Fine-tune Prompts**:   - Edit system and user prompts in `_build_system_prompt()` and `_build_user_prompt()`4. **Integrate with Real Systems**:   - Zendesk, Intercom, Freshdesk via their APIs   - Custom CRM or support platform### 📞 SUPPORTFor issues or questions:1. Check `DEPLOYMENT.md` troubleshooting section2. Review LangChain 1.2.0 documentation: https://python.langchain.com/3. Check LangGraph docs: https://langchain-ai.github.io/langgraph/4. Groq API docs: https://console.groq.com/docs### 📝 LICENSE & ATTRIBUTIONThis notebook is provided as educational material for building AI applications with LangChain 1.2.0+.---**Notebook Created**: 2026-05-28 10:19:59**LangChain Version**: 1.2.0+**Status**: Production Ready ✅

## 🎉 YOU'RE ALL SET!### Next Actions:1. **🔐 Set API Keys** in Colab Secrets (🔑 icon):   - `GROQ_API_KEY`: Get from https://console.groq.com   - `TAVILY_API_KEY`: Optional, from https://tavily.com2. **▶️ Run All Cells** in order from Phase 1 to Phase 93. **🧪 Test the System** with Phase 9 example4. **📦 Download Files** at the end (all .py files zipped)5. **🚀 Deploy** using Phase 10 deployment guide6. **📚 Customize** with your own tools, knowledge base, and prompts---### 🎯 Common Use Cases:- **Support Ticket Auto-response**: Generate professional responses instantly- **Knowledge Base Q&A**: Semantic search over company docs- **Customer Context**: Multi-scope memory (customer + company)- **Tool Integration**: Call APIs for customer lookups, billing checks- **Escalation Logic**: Detect high-priority issues and route appropriately---### 💬 Questions?- **LangChain Docs**: https://python.langchain.com/- **Groq Console**: https://console.groq.com/- **GitHub**: Share your extensions and integrations!---**Happy Building! 🚀**